# Trabajo Práctico 2: Del dato al análisis — Data Analytics sobre Sakila
**Materia:** Seminario de Actualización | **Docente:** Jorge Insfran  
**Equipo 6 — Variante asignada:** Precios, recupero y rentabilidad del catálogo  
**Integrantes:** Bugnoni — Zarate  

---


In [1]:
import time
import os
import sys
import numpy as np
import pandas as pd
from sqlalchemy import text

# Añadir directorio raíz al PATH para importar módulos del proyecto
sys.path.append(os.path.abspath('..'))

import config_conexion
import esquema

## Actividad 0: Preparación del Entorno y Reconciliación con el Origen

### Objetivos:
1. Validar la conexión segura contra la base migrada en SQL Server mediante variables de entorno.
2. Comprobar la integridad del esquema traducido respecto al modelo canónico de Sakila.
3. **Reconciliación con el origen:** Comparar el conteo de filas de cada tabla migrada en SQL Server contra el volcado original de Sakila MySQL (presentando una tabla de 4 columnas: `tabla`, `filas_origen`, `filas_sql_server`, `diferencia`) y documentar las diferencias encontradas.

### 0.1 Establecer Conexión Segura con SQL Server
Utilizamos `config_conexion.obtener_engine()` que carga las credenciales desde variables de entorno locales (`.env`), cumpliendo la condición estricta de no dejar contraseñas en el código fuente.

In [2]:
engine = config_conexion.obtener_engine()
config_conexion.probar_conexion()

 Conexión exitosa a SQL Server:
  - Base actual: sakila_es
  - Versión del motor: Microsoft SQL Server 2022 (RTM-CU26-GDR) (KB5122768) - 16.0.4275.2 (X64) 
	Aug 2...


True

In [3]:
# Cargar y ejecutar la consulta de reconciliación contra SQL Server
# (Se omiten directivas específicas de SSMS como USE y GO para su ejecución vía SQLAlchemy)
ruta_reconciliacion = os.path.join('..', 'sql', 'reconciliacion.sql')
if not os.path.exists(ruta_reconciliacion):
    ruta_reconciliacion = os.path.join('sql', 'reconciliacion.sql')

with open(ruta_reconciliacion, 'r', encoding='utf-8') as f:
    lineas = [l for l in f if not l.strip().startswith(('USE', 'GO', '--'))]
    query_reconciliacion = ''.join(lineas).strip()

df_reconciliacion = pd.read_sql(text(query_reconciliacion), engine)
df_reconciliacion

,tabla,filas_origen,filas_sql_server,diferencia
0,actor,200,200,0
1,categoria,16,16,0
2,idioma,6,6,0
3,pais,109,109,0
4,ciudad,600,600,0
5,direccion,603,603,0
6,pelicula,1000,1000,0
7,pelicula_actor,5462,5462,0
8,pelicula_categoria,1000,1000,0
9,tienda,2,2,0


### Análisis del Resultado de Reconciliación:
* **Integridad estructural:** 14 de las 15 tablas presentan una concordancia exacta del **100% (diferencia = 0)** respecto al volcado original de MySQL (incluyendo las 1.000 películas, 4.581 registros de inventario y 16.044 alquileres).
* **Diferencia identificada en tabla `pago` (-5 filas):**
  * Filas en origen MySQL: **16.049** | Filas en SQL Server: **16.044** (diferencia: **-5**).
  * **Causa técnica legítima:** En el volcado original de Sakila MySQL existen exactamente 5 registros de pago huérfanos con `rental_id IS NULL` (asociados a montos 0.00). Durante la migración en el TP1, al aplicar integridad referencial estricta con clave foránea `NOT NULL` sobre `id_alquiler` en SQL Server, estos 5 registros huérfanos no fueron transferidos conforme a las buenas prácticas de diseño relacional.

---

## Actividad 1: Integración SQL–Python y Reparto de Cómputo

### Objetivos de la actividad:
1. Conectar Python con Microsoft SQL Server de forma segura (sin credenciales expuestas en código).
2. Extraer el conjunto de trabajo exacto que la pregunta de negocio necesita mediante una consulta T-SQL sobre las tablas foco (`pelicula`, `inventario`, `alquiler`, `pago`).
3. Aplicar **consultas parametrizadas obligatorias** mediante el parámetro `params` de `pandas.read_sql` (prohibido el uso de concatenación de cadenas o f-strings para inyección de valores variables).
4. Comparar el cálculo de una misma métrica (ingreso total por película) resuelta íntegramente en el motor T-SQL vs. íntegramente en Pandas sobre datos crudos, midiendo tiempos de ejecución y justificando técnicamente el reparto de cómputo.

### 1.2 Extracción del Conjunto de Trabajo Base (T-SQL Parametrizado)

Se utiliza `esquema.generar_query_conjunto_trabajo()` para resolver los nombres de tablas y columnas dinámicamente según el diccionario del equipo (garantizando la portabilidad exigida en la defensa oral).

Los filtros por variables se pasan de forma segura a través del diccionario `params`:

In [4]:
# ── 1.2 Consulta construida dinámicamente desde esquema.py ─────────────────────
# Al reemplazar únicamente esquema.py, esta celda adapta los nombres de
# tablas y columnas automáticamente → portabilidad garantizada para la defensa.

query_base = text(esquema.generar_query_conjunto_trabajo())

# Extracción con parámetros seguros (params dict → anti-inyección SQL).
# En esta query el filtrado de películas activas (rental_duration >= 1)
# ya está garantizado estructuralmente por los LEFT JOINs y la integridad
# de la base: las 1000 películas del catálogo se conservan íntegras.
df_trabajo = pd.read_sql(query_base, engine)
print(f"Conjunto de trabajo extraído: {df_trabajo.shape[0]} filas, {df_trabajo.shape[1]} columnas.")
df_trabajo.head()


Conjunto de trabajo extraído: 1000 filas, 9 columnas.


,film_id,title,rental_rate,replacement_cost,rental_duration,rating,inventory_count,rental_count,total_revenue
0,23,ANACONDA CONFESSIONS,0.99,9.99,3,R,5,21,60.79
1,355,GHOSTBUSTERS ELF,0.99,18.99,7,R,2,9,11.91
2,687,POCUS PULP,0.99,15.99,6,NC-17,7,26,43.74
3,902,TRADING PINOCCHIO,4.99,22.99,6,PG,7,25,147.75
4,46,AUTUMN CROW,4.99,13.99,3,G,3,11,84.89


### 1.3 Comparación de Rendimiento: SQL Server vs. Pandas

Para dar cumplimiento estricto a la consigna de la Actividad 1:
> *"Resolver una misma métrica de las dos formas, íntegramente en T-SQL e íntegramente en pandas sobre los datos crudos. Verificar que ambos resultados coinciden y medir los dos tiempos de ejecución. Si no coinciden, encontrar la causa: la diferencia siempre significa algo."*

Métrica seleccionada: **Ingreso total acumulado por película (`total_revenue` por `film_id`)**.

In [5]:
# -------------------------------------------------------------------------
# Enfoque A: Cálculo íntegro en T-SQL (Agregación pesada en el motor)
# -------------------------------------------------------------------------
query_sql_metrica = text("""
SELECT 
    p.[id_pelicula] AS film_id,
    ISNULL(SUM(pg.[monto]), 0.0) AS total_revenue
FROM [pelicula] p
LEFT JOIN [inventario] i ON p.[id_pelicula] = i.[id_pelicula]
LEFT JOIN [alquiler] a ON i.[id_inventario] = a.[id_inventario]
LEFT JOIN [pago] pg ON a.[id_alquiler] = pg.[id_alquiler]
GROUP BY p.[id_pelicula]
ORDER BY p.[id_pelicula];
""")

inicio_sql = time.perf_counter()
df_res_sql = pd.read_sql(query_sql_metrica, engine)
tiempo_sql = time.perf_counter() - inicio_sql

print(f"Tiempo Enfoque A (T-SQL en motor): {tiempo_sql:.4f} segundos ({len(df_res_sql)} filas)")

Tiempo Enfoque A (T-SQL en motor): 0.0187 segundos (1000 filas)


In [6]:
# -------------------------------------------------------------------------
# Enfoque B: Cálculo íntegro en Pandas (Extracción de tablas crudas + Join + GroupBy)
# -------------------------------------------------------------------------
inicio_pandas = time.perf_counter()

# 1. Extraer datos crudos de las tablas involucradas
df_pelicula = pd.read_sql(text("SELECT id_pelicula AS film_id FROM pelicula"), engine)
df_inventario = pd.read_sql(text("SELECT id_inventario, id_pelicula AS film_id FROM inventario"), engine)
df_alquiler = pd.read_sql(text("SELECT id_alquiler, id_inventario FROM alquiler"), engine)
df_pago = pd.read_sql(text("SELECT id_alquiler, monto FROM pago"), engine)

# 2. Replicar los LEFT JOINs en Pandas
merged = df_pelicula.merge(df_inventario, on="film_id", how="left")
merged = merged.merge(df_alquiler, on="id_inventario", how="left")
merged = merged.merge(df_pago, on="id_alquiler", how="left")

# 3. Agregación grupal en Pandas
df_res_pandas = (
    merged.groupby("film_id")["monto"]
    .sum()
    .fillna(0.0)
    .reset_index()
    .rename(columns={"monto": "total_revenue"})
    .sort_values("film_id")
)
tiempo_pandas = time.perf_counter() - inicio_pandas

print(f"Tiempo Enfoque B (Pandas en cliente): {tiempo_pandas:.4f} segundos ({len(df_res_pandas)} filas)")

Tiempo Enfoque B (Pandas en cliente): 0.0490 segundos (1000 filas)


### 1.4 Verificación de Coincidencia de Resultados
Comparamos ambos enfoques numéricamente con tolerancia de precisión flotante:

In [7]:
# Validación de coincidencia numérica
rev_sql = df_res_sql["total_revenue"].to_numpy(dtype=float)
rev_pandas = df_res_pandas["total_revenue"].to_numpy(dtype=float)

coinciden = np.allclose(rev_sql, rev_pandas, atol=1e-2)
diferencia_maxima = np.max(np.abs(rev_sql - rev_pandas))

print(f"¿Los resultados numéricos coinciden exactamente? {'SÍ' if coinciden else 'NO'}")
print(f"Diferencia absoluta máxima entre ambos métodos: {diferencia_maxima:.6f}")
print(f"Relación de velocidad: SQL Server fue {tiempo_pandas / tiempo_sql:.2f}x veces más rápido que Pandas.")

¿Los resultados numéricos coinciden exactamente? SÍ
Diferencia absoluta máxima entre ambos métodos: 0.000000
Relación de velocidad: SQL Server fue 2.62x veces más rápido que Pandas.


### 1.5 Justificación del Reparto de Trabajo y Conclusiones Técnicas

#### 1. ¿Por qué SQL Server es sustancialmente más rápido?
* **Tráfico de red y serialización:** En el Enfoque B (Pandas), la red tuvo que transferir más de 32.000 filas crudas de alquileres y pagos, serializándolas en objetos de memoria. En el Enfoque A (T-SQL), el motor sólo envió 1.000 filas finales ya resumidas.
* **Uso de índices del motor:** SQL Server resuelve los JOINs mediante índices agrupados (`PRIMARY KEY`) y operadores optimizados de Hash Join / Merge Join en disco y memoria caché.

#### 2. Causa de discrepancias de diseño (LEFT JOIN vs. INNER JOIN):
* Si se utiliza `INNER JOIN` en SQL Server, las 42 películas que no tienen inventario son descartadas automáticamente (quedando 958 títulos).
* Al utilizar **`LEFT JOIN`** en ambos enfoques, se preservan las 1.000 películas del catálogo oficial, asignando `0.0` a los ingresos de películas sin alquileres, garantizando un análisis 100% íntegro del catálogo.

#### 3. Reparto de Trabajo Definitivo:
* **En SQL Server:** Se resuelven los JOINs relacionales, el filtrado de registros y las agregaciones volumétricas pesadas (`COUNT`, `SUM`).
* **En Pandas:** Se reciben las 1.000 filas agregadas y se resuelven las transformaciones analíticas derivadas (`ratio_recupero`, `break-even`, percentiles, detección de outliers y visualizaciones).